In [1]:
import os
import requests
from pathlib import Path
import random
from typing import List, Tuple
import zipfile
from zipfile import ZipFile

import torch
from torch import nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

import torchvision
from torchvision import datasets, transforms
from torchvision.transforms import ToTensor
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt

try:
  from torchinfo import summary
except:
  !pip install -q torchinfo
  from torchinfo import summary

try:
  from torchmetrics import Accuracy
except:
  !pip install -q torchmetrics
  from torchmetrics import Accuracy

try:
  import tqdm.auto as tqdm
except:
  !pip install -q tqdm
  import tqdm.auto as tqdm

device = torch.device("mps"
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print("All imported")
print(f"Device: {device}")

/Users/admin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


All imported
Device: mps


In [2]:
def set_seeds(seed: int=42):
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)

In [3]:
data_path = Path("data")
image_path = data_path / "pizza_steak_sushi"
zip_path = data_path / "pizza_steak_sushi.zip"

if image_path.is_dir():
  print(f"[INFO] {image_path} directory exists, skipping download.")
else:
  print(f"[INFO] Did not find {image_path} directory, creating one...")
  image_path.mkdir(parents=True, exist_ok=True)
  with open(zip_path, "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print(f"[INFO] Downloading {image_path.name} from github")
    f.write(request.content)
  with zipfile.ZipFile(zip_path, "r") as zip_ref:
    print(f"[INFO] Unzipping {image_path.name} data...")
    zip_ref.extractall(image_path)
    os.remove(zip_path)

[INFO] data/pizza_steak_sushi directory exists, skipping download.


In [4]:
train_data = image_path / "train"
test_data = image_path / "test"

train_data, test_data

(PosixPath('data/pizza_steak_sushi/train'),
 PosixPath('data/pizza_steak_sushi/test'))

In [5]:
# normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                                  std=[0.229, 0.224, 0.225])

# manual_transforms = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     normalize
# ])

weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT 
manual_transforms = weights.transforms()

print(f"Manually created transforms: {manual_transforms}")

train_dataset = datasets.ImageFolder(root=train_data,
                                     transform=manual_transforms,
                                     target_transform=None)
test_dataset = datasets.ImageFolder(root=test_data,
                                    transform=manual_transforms,
                                    target_transform=None)

class_names = train_dataset.classes
    
print("\n", len(train_dataset), len(test_dataset))
print(f"\n{class_names}")

Manually created transforms: ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

 225 75

['pizza', 'steak', 'sushi']


In [6]:
BATCH_SIZE = 32

train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=BATCH_SIZE,
                              shuffle=True)
test_dataloader = DataLoader(dataset=test_dataset,
                             batch_size=BATCH_SIZE,
                             shuffle=False)

train_dataloader, test_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x11d0d7c70>,
 <torch.utils.data.dataloader.DataLoader at 0x11d0d75e0>)

In [8]:
model = torchvision.models.efficientnet_b0(weights=weights).to(device)
model

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [11]:
for param in model.features.parameters():
    param.requeries_grad = False 

set_seeds()

model.classifier = torch.nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280,
              out_features=len(class_names),
              bias=True)).to(device)

In [13]:
summary(model=model,
        input_size=(32, 3, 224, 224),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"]
        )

Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [32, 3, 224, 224]    [32, 3]              --                   True
├─Sequential (features)                                      [32, 3, 224, 224]    [32, 1280, 7, 7]     --                   True
│    └─Conv2dNormActivation (0)                              [32, 3, 224, 224]    [32, 32, 112, 112]   --                   True
│    │    └─Conv2d (0)                                       [32, 3, 224, 224]    [32, 32, 112, 112]   864                  True
│    │    └─BatchNorm2d (1)                                  [32, 32, 112, 112]   [32, 32, 112, 112]   64                   True
│    │    └─SiLU (2)                                         [32, 32, 112, 112]   [32, 32, 112, 112]   --                   --
│    └─Sequential (1)                                        [32, 32, 112, 112]   [32, 16, 112

In [16]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.001)

In [19]:
try:
    from torch.utils.tensorboard import SummaryWriter 
except:
    print("[INFO] Couldn`t find tensorboard... installing it.")
    !pip install -q tensorboard 
    from torch.utils.tensorboard import SummaryWriter 

writer = SummaryWriter()
writer